In [ ]:
# import all relevant packages
import pandas as pd
from pandas.io import gbq
import requests
import json

In [ ]:
# call cryptocompare historical stats API

# api_key should be yours
def get_data():
    crypto_api_url = "https://data-api.coindesk.com/index/cc/v1/historical/days"
    ### you would change below from 10 -> 5,000 to load the last 5,000 days
    ### you would change BTC-USD to ETH-USD to get Ethereum USD prices opposed to Bitcoin USD (BTC-USD)
    payload = {'market': 'cadli','instrument': 'ETH-USD', 'aggregate': '1','fill': 'true','apply_mapping': 'true','limit': '5000', 'response_format':'JSON', 'api_key': '##FILL API KEY HERE###'}
    request_data = requests.get(crypto_api_url,params=payload)
    #return request_data.json()['Data']['Data']
    return request_data.json()['Data']

CryptoAPIDataResponse = get_data()

#CryptoAPIDataResponse

In [ ]:
#convert to JSON
CryptoAPIDataResponseJSON = json.dumps(CryptoAPIDataResponse)
print(CryptoAPIDataResponseJSON)

In [ ]:
#convert to pandas dataframe to prepare for insertion into BigQuery
datalistDF = pd.read_json(CryptoAPIDataResponseJSON)
datalistDF.head()

In [ ]:
# Load data to BigQuery using service account credentials

from google.oauth2 import service_account

# SERVICE ACCOUNT SETUP:
# 1. In Google Cloud Console, go to IAM & Admin > Service Accounts.
# 2. Select or create a service account for your project.
# 3. Go to Keys > Add Key > Create New Key.
# 4. Select JSON and download the credentials file.
# 5. Open the downloaded JSON file.
# 6. Copy the entire contents and paste them between the parentheses below.
#
# IMPORTANT: Never commit your actual service account credentials to GitHub.

credentials = service_account.Credentials.from_service_account_info(
    ### INSERT JSON FROM SERVICE ACCOUNT HERE ###
)

datalistDF.to_gbq(
    destination_table='crypto_dataset.crypto_history_ETH',
    project_id='osu-demo-project-2026-508513',
    if_exists='fail',
    credentials=credentials
)